# Stitch viewer: tiles, design, contours and design errors, before and after stitching

Runs the pipeline on the chosen tiles, prints the results of each step, and at the end opens
**one** interactive window (pyqtgraph, OpenGL) with a tab per viewer:

1. **Per tile** (`process_tile`): load the SEM image, detect contacts (Otsu), refine the contours
   to the maximum gradient, read the design contacts from the tile's `.oas` in the tone (normal /
   reversed) that matches the SEM contacts. Tiles where neither tone matches are flagged.
   Design and SEM are matched by first finding the shift at which most contacts line up
   (up to `DESIGN_SEARCH_NM`), then pairing within `DESIGN_TOLERANCE_NM` after it. The shift
   only chooses the pairs; the error keeps it. For the design errors (section 3) all tiles'
   shifts are chosen together, so that neighbouring tiles agree.
2. **Stitching**, translation per tile, failed overlaps and unstitched tiles listed:
   the SEM tiles (`stitch_tiles`) and, the same way, the per-tile design files (`stitch_design`),
   which can be offset against each other.
3. **Design errors** (`design_errors`) for each placement (`placement_corrections`): nominal,
   stitched (corrections average 0), stitched with the first tile fixed. Raw error SEM − design,
   no affine removed yet. Tiles that could not be measured are listed with the reason. If
   stitching or the design errors failed, the stitch viewer opens right there (section 3b).
   **Affine stitching** (2b, 3c): the same overlap ties, one full affine per SEM tile
   (`stitch_tiles_affine`), and its design errors, for comparison with the translation stitching.
4. **RANSAC** global affine, **moving-window** affine and **row pitch** on the merged contacts
   (translation stitching), then RANSAC and moving window on the affine-stitched contacts and both
   stitchings side by side (section 9).
5. **Analysis window** (`StitchingComparison`, section 10): stitching viewer (translation), the
   comparison, and the full analysis (RANSAC monitor, registration error, moving-window tuner, row
   pitch) of each stitching.

The window blocks this notebook until it is closed. Positions in µm, errors in nm.

In [ ]:
DATA_DIR = r"A:\Stitch_monitor\In_cell_without_PTM_20260914\260910_1143_Norm"
TILES = None              # None = all tiles, or a list of indices in read_tile_index order, e.g. list(range(10))
LAYER = (100, 0)          # (layer, datatype) of the contacts in the .oas
DETECTION = "band"        # "otsu" (plain), "otsu3" (3 classes, threshold hole | background), "band" (regions enclosed by the bright band)
SEARCH_PX = 5.0           # refinement searches this many pixels in/out along the contour normal
DESIGN_SEARCH_NM = 100.0  # design-SEM matching: largest expected shift of a tile's SEM contacts against its design (also > pitch / 2)
DESIGN_TOLERANCE_NM = 10.0  # design-SEM matching: pair contacts within this after removing the tile shift (< pitch / 2)
MIN_MATCH_FRACTION = 0.5  # a design tone is accepted if at least this fraction of SEM contacts match it
STITCH_REFINED = True     # stitch with the refined centres (False: Otsu centres)
MAX_MATCH_NM = 25.0       # SEM overlaps: pair contacts closer than this, the largest B - A stage offset (< pitch / 2)
DESIGN_MAX_MATCH_NM = 25.0  # design overlaps: the largest offset between neighbouring .oas files (< pitch / 2)
OUTLIER_FACTOR = 3.0      # overlap outlier if residual > this x the median residual
MIN_MATCHED = 5           # pairs with fewer matched contacts are not used (reported below)
GROUP_RADIUS_NM = 5.0     # observations from different tiles whose stitched design positions are this close = one contact
SPREAD_FLAG_NM = 2.0      # flag merged contacts whose SEM observations differ by more than this
ARROW_SCALE = 10          # error-map lines are drawn this many times longer than the error
USE_OPENGL = True         # set False if the viewer window stays black
DIAGNOSTIC_VIEWER = "auto"  # stitch viewer after section 3: "auto" = only if stitching or the design errors failed, True = always, False = never
RANSAC_PLACEMENT = "mean"   # RANSAC on the merged contacts of this placement: "mean" (stitched) or "first"
RANSAC_THRESHOLD_NM = 0.5   # RANSAC inlier if |G(SEM) - design| < this
RANSAC_SEED = 0             # RANSAC random samples are reproducible for a given seed
RANSAC_DELAY_MS = 300       # RANSAC monitor: pause between steps
ROW_GAP_NM = 10.0           # registration error per row: a new row starts where sorted design y jumps by more than this
MOVING_WINDOW_UM = 40.0     # moving-window affine: window height along y
MOVING_STEP_UM = 5.0        # moving-window affine: the window moves up this much each step

In [ ]:
import time

import numpy as np
import pandas as pd
import pyqtgraph as pg

from affine_ransac.fitting.moving_window import moving_window_affine
from affine_ransac.fitting.ransac import ransac_affine
from affine_ransac.geometry.affine import recentre, report_terms
from affine_ransac.io.design import load_layout, read_polygons
from affine_ransac.io.metadata import read_tile_index
from affine_ransac.pipeline import (process_tile, stitch_design, stitch_tiles, stitch_tiles_affine, tie_residuals,
                                    tile_neighbours)
from affine_ransac.registration import (design_errors, error_summary, merge_observations, row_means, row_pitch,
                                        stitching_summary)
from affine_ransac.stitching import placement_corrections
from affine_ransac.view_compare import StitchingComparison
from affine_ransac.view_stitch import StitchViewer

## 1. Load the tiles and detect their contacts

In [ ]:
all_tiles = read_tile_index(DATA_DIR)
records = all_tiles if TILES is None else [all_tiles[i] for i in TILES]
tile_ids = [r.tile_id for r in records]

start = time.perf_counter()
tiles = [
    process_tile(r.image_path, r.oas_path, (r.center_x_nm, r.center_y_nm), (r.fov_x_nm, r.fov_y_nm),
                 LAYER, SEARCH_PX, DESIGN_SEARCH_NM, MIN_MATCH_FRACTION, DETECTION, DESIGN_TOLERANCE_NM)
    for r in records
]
print(f"{len(tiles)} of {len(all_tiles)} tiles in {time.perf_counter() - start:.1f} s")

per_tile = pd.DataFrame({
    "tile": tile_ids,
    "design": [len(t.design.centers) for t in tiles],
    "otsu": [len(t.otsu.centers) for t in tiles],
    "refined": [len(t.refined.centers) for t in tiles],
    "tone": ["reversed" if t.design.reversed else "normal" for t in tiles],
    "match_fraction": [t.design.match_fraction for t in tiles],
    "design_ok": [t.design.ok for t in tiles],
    "shift_x_nm": [t.design.shift_nm[0] for t in tiles],  # SEM - design shift found by the tone check
    "shift_y_nm": [t.design.shift_nm[1] for t in tiles],
    "ambiguous": [t.design.ambiguous for t in tiles],      # shift chosen by the tie-break (smallest of near-equal)
})
print(f"Design tone: {(per_tile.tone == 'normal').sum()} normal, {(per_tile.tone == 'reversed').sum()} reversed")
flagged = per_tile[~per_tile.design_ok]
if len(flagged):
    print(f"\n!! {len(flagged)} tile(s) FLAGGED: neither design tone matches the SEM (or no .oas):")
    display(flagged.round(2))
per_tile.describe().round(2)

## 2. Stitching: SEM tiles and design files

In [ ]:
centers = np.array([t.center_nm for t in tiles])
fovs = np.array([t.fov_nm for t in tiles])
sem_points = [t.sem_points_nm(STITCH_REFINED) for t in tiles]
stitch = stitch_tiles(sem_points, centers, fovs, MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED)
design_stitch = stitch_design(tiles, DESIGN_MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED)

for name, result in (("SEM", stitch), ("Design", design_stitch)):
    failed = [r for r in result.rejected if r.failed]
    print(f"{name}: {len(result.pairs)} overlapping pairs used, {len(failed)} not matched, "
          f"{len(result.rejected) - len(failed)} skipped (fewer than {MIN_MATCHED} contacts in the overlap); "
          f"corrections |x| max {np.nanmax(np.abs(result.corrections[:, 0])):.2f} nm, "
          f"|y| max {np.nanmax(np.abs(result.corrections[:, 1])):.2f} nm")
    if failed or len(result.unplaced):
        print(f"!! {name.upper()} STITCHING INCOMPLETE. Tiles not stitched: {[tile_ids[k] for k in result.unplaced]}")
        display(pd.DataFrame([{"tile_a": tile_ids[r.i], "tile_b": tile_ids[r.j], "contacts_in_overlap": r.in_box,
                               "matched": r.matched} for r in failed]))

pd.DataFrame({
    "tile": tile_ids,
    "sem_correction_x_nm": stitch.corrections[:, 0], "sem_correction_y_nm": stitch.corrections[:, 1],
    "design_correction_x_nm": design_stitch.corrections[:, 0], "design_correction_y_nm": design_stitch.corrections[:, 1],
}).round(2)

### 2b. Affine stitching: one affine per tile

The same overlap pairs and tie contacts (each pair's inliers) as the translation stitching above,
but each tile gets a full affine `T_k(p) = p + D_k (p − c_k) + t_k` about its centre `c_k`:
shift, scale (Mx, My), rotation and orthogonality (`stitch_tiles_affine`). Gauge: the D_k and t_k
average zero (mean tile affine = identity); any common affine is taken up by the RANSAC affine
later. Tiles whose tie contacts lie on one line are warned about (their scale across it stays
nominal). Only the SEM is stitched with affines; the design keeps its translation stitching.

The first lines compare the **tie residuals** T_i(a) − T_j(b) of the overlap contacts after both
stitchings (how well the tiles agree). The table lists the correction terms per tile (`report_terms`
about the tile centre: Tx, Ty = shift at the centre, nm; Mx, My in ppm; rotation, orthogonality in
degrees); the summary first.

In [ ]:
affines = stitch_tiles_affine(sem_points, centers, stitch)  # (n_tiles, 3, 3) nominal -> stitched, NaN = not stitched
ties = {"translation": tie_residuals(sem_points, stitch, stitch.corrections),
        "affine": tie_residuals(sem_points, stitch, affines)}
for name, r in ties.items():
    length = np.linalg.norm(r, axis=1)
    print(f"Overlap ties after {name} stitching: {len(r)} contacts, RMS {np.sqrt((length ** 2).mean()):.3f} nm, "
          f"max {length.max():.3f} nm")

tile_terms = pd.DataFrame([{label: value for label, value, _ in report_terms(recentre(a, c), p - c)}
                           for a, c, p in zip(affines, centers, sem_points)], index=tile_ids)
display(tile_terms.describe().round(4))
tile_terms.round(4)

## 3. Design errors per placement (raw: SEM − design, no affine removed)

Each tile is matched to its own design, so a contact in an overlap is first observed once per
tile (2 in a strip, up to 4 in a corner). `merge_observations` then turns the observations into one
entry per physical contact: observations from different tiles whose **stitched design** positions
are within `GROUP_RADIUS_NM` are averaged. Their **spread** (largest distance between the SEM
observations) is kept; a large spread means the tiles disagree about that contact.

**Matching** (`design_errors` with `neighbours`): for each tile, the candidate shifts at which many
SEM contacts land on a design contact (up to `DESIGN_SEARCH_NM`), each scored by its **mismatches**:
inside the area where both are complete, the contacts of either side without a partner within
`DESIGN_TOLERANCE_NM`. At the true shift that is ~0; one row off, every missing contact of the
pattern leaves contacts without a partner. A candidate with clearly fewer mismatches wins outright
(neighbours cannot overrule it). Only tiles where candidates tie (*ambiguous*: no missing contact
decides, fully periodic content) follow their overlapping neighbours (`consistent_choice`); a group
of tied tiles without any decided tile takes the smallest median shift. Contacts are paired after
removing the shift, but each error is the full SEM − design. The table lists per tile (stitched
placement) the shift, **mismatches** at it (~0 expected; more = failed detections or a wrong row)
and **neighbour_jump_nm**, the largest difference from a neighbour's shift (about one pitch = a
seam where tiles were paired one row apart).

The common offset depends on the placement, so look at the pattern: a uniform shift, a rotation
(arrows circling) or a scale (arrows pointing outward/inward).

In [ ]:
design_points = [t.design_points_nm() for t in tiles]
design_ok = [t.design.ok for t in tiles]
neighbours = tile_neighbours(centers, fovs)  # overlapping tiles: their matching shifts must agree
observations, merged = {}, {}
for placement, (sem_corrections, design_corrections) in placement_corrections(stitch.corrections,
                                                                               design_stitch.corrections).items():
    observations[placement] = design_errors(sem_points, design_points, sem_corrections, design_corrections,
                                            design_ok, DESIGN_SEARCH_NM, DESIGN_TOLERANCE_NM, neighbours)
    merged[placement] = merge_observations(observations[placement], design_stitch.corrections, GROUP_RADIUS_NM)

for placement, e in observations.items():
    if e.skipped:
        print(f"!! {placement}: {len(e.skipped)} tile(s) NOT MEASURED (no error computed):")
        display(pd.DataFrame([{"tile": tile_ids[k], "reason": reason} for k, reason in e.skipped.items()]))

e = observations["mean"]
jumps = e.neighbour_jump_nm
print(f"Matching shift (stitched): {np.sum(~np.isnan(e.match_shift_nm[:, 0]))} tiles matched, "
      f"{e.ambiguous.sum()} tied (no missing contact decides: chosen by the neighbours); "
      f"largest neighbour jump {np.nanmax(jumps) if np.isfinite(jumps).any() else float('nan'):.1f} nm")
if np.nanmax(jumps, initial=0) > 2 * DESIGN_TOLERANCE_NM:
    print(f"!! {np.sum(jumps > 2 * DESIGN_TOLERANCE_NM)} tile(s) differ from a neighbour by more than "
          f"{2 * DESIGN_TOLERANCE_NM:g} nm: check them in the stitch viewer")
display(pd.DataFrame({"tile": tile_ids, "shift_x_nm": e.match_shift_nm[:, 0], "shift_y_nm": e.match_shift_nm[:, 1],
                      "ambiguous": e.ambiguous, "mismatches": e.match_mismatches,
                      "neighbour_jump_nm": jumps}).round(2))

m = merged["mean"]
sizes = pd.Series(m.count).value_counts().sort_index()
print(f"{len(observations['mean'].tile)} observations -> {len(m.count)} contacts; "
      f"observations per contact: {({int(k): int(v) for k, v in sizes.items()})}")
overlap = m.count > 1
if overlap.any():
    print(f"Spread of merged overlap contacts (stitched): median {np.median(m.spread_nm[overlap]):.2f} nm, "
          f"max {m.spread_nm.max():.2f} nm; {(m.spread_nm > SPREAD_FLAG_NM).sum()} above {SPREAD_FLAG_NM} nm")
    worst = np.argsort(m.spread_nm)[::-1][:10]
    display(pd.DataFrame({
        "x_um": m.design_nm[worst, 0] / 1000, "y_um": m.design_nm[worst, 1] / 1000,
        "spread_nm": m.spread_nm[worst], "observations": m.count[worst],
        "tiles": [[tile_ids[k] for k in observations["mean"].tile[m.members[g]]] for g in worst],
    }).round(4))

pd.DataFrame({placement: error_summary(e.error_nm) for placement, e in merged.items()}).round(2)

### 3b. Diagnostic viewer (opens only if something failed)

With `DIAGNOSTIC_VIEWER = "auto"` the stitch viewer opens here when a stitching pair did not
match, a tile could not be stitched, a tile was not measured in some placement, or a tile's matching
shift jumps against a neighbour by more than 2 × `DESIGN_TOLERANCE_NM`, so the
failure can be inspected before the later steps (which need measured contacts). Compare the
placements with the buttons: tiles that fail only when stitched point at the stitching
corrections, tiles that fail at the nominal placement too point at the SEM or design contacts
themselves (turn on the design centres and the refined contours). Not-measured tiles are framed
in magenta in the error map. Closes with the window. Without failures it is the *Stitching* tab
of the analysis window (section 10).

In [ ]:
design_polygons = [read_polygons(load_layout(r.oas_path), *LAYER) if r.oas_path else [] for r in records]

problems = [f"{name} stitching: {sum(r.failed for r in result.rejected)} overlap(s) not matched, "
            f"{len(result.unplaced)} tile(s) not stitched"
            for name, result in (("SEM", stitch), ("design", design_stitch))
            if len(result.unplaced) or any(r.failed for r in result.rejected)]
problems += [f"{placement}: {len(e.skipped)} of {len(tiles)} tile(s) not measured"
             for placement, e in observations.items() if e.skipped]
problems += [f"{placement}: {np.sum(e.neighbour_jump_nm > 2 * DESIGN_TOLERANCE_NM)} tile(s) whose matching shift "
             f"jumps by more than {2 * DESIGN_TOLERANCE_NM:g} nm against a neighbour"
             for placement, e in observations.items() if np.nanmax(e.neighbour_jump_nm, initial=0) > 2 * DESIGN_TOLERANCE_NM]

if DIAGNOSTIC_VIEWER is True or (DIAGNOSTIC_VIEWER == "auto" and problems):
    print("Opening the stitch viewer: " + ("; ".join(problems) or "DIAGNOSTIC_VIEWER = True"))
    app = pg.mkQApp("Stitch viewer")
    diagnostic_view = StitchViewer(tiles, tile_ids, design_polygons, stitch, design_stitch,
                                   refined=STITCH_REFINED, use_opengl=USE_OPENGL, errors=merged,
                                   arrow_scale=ARROW_SCALE, spread_flag_nm=SPREAD_FLAG_NM)
    diagnostic_view.show()
    app.exec()
else:
    print("No stitching or measurement failures; the stitch viewer is the Stitching tab in section 9.")

### 3c. Design errors with the affine stitching

The same design errors (`design_errors` with the neighbours, `merge_observations`) with each SEM
tile placed by its affine (section 2b) instead of a shift; the design keeps its translation
stitching. Only the stitched placement (mean gauge). Compare the spread of the merged overlap
contacts and the matching with section 3. The stitch viewer (3b, *Stitching* tab) shows the
translation stitching only.

In [ ]:
affine_observations = design_errors(sem_points, design_points, affines, design_stitch.corrections, design_ok,
                                    DESIGN_SEARCH_NM, DESIGN_TOLERANCE_NM, neighbours)
affine_contacts = merge_observations(affine_observations, design_stitch.corrections, GROUP_RADIUS_NM)
if affine_observations.skipped:
    print(f"!! affine: {len(affine_observations.skipped)} tile(s) NOT MEASURED (no error computed):")
    display(pd.DataFrame([{"tile": tile_ids[k], "reason": reason} for k, reason in affine_observations.skipped.items()]))
for name, m in (("translation (mean)", merged["mean"]), ("affine", affine_contacts)):
    overlap = m.count > 1
    if overlap.any():
        print(f"{name}: {len(m.count)} contacts; spread of merged overlap contacts median "
              f"{np.median(m.spread_nm[overlap]):.3f} nm, max {m.spread_nm.max():.3f} nm, "
              f"{(m.spread_nm > SPREAD_FLAG_NM).sum()} above {SPREAD_FLAG_NM} nm")
pd.DataFrame({"translation (mean)": error_summary(merged["mean"].error_nm),
              "affine": error_summary(affine_contacts.error_nm)}).round(2)

## 5. RANSAC global affine (on the merged contacts)

`ransac_affine` finds one affine G for the whole field that maps the stitched SEM centres onto the
design centres: random 3-point samples, the model with the most inliers (residual <
`RANSAC_THRESHOLD_NM`), then a least-squares refit on its inliers, all relative to the design
centroid. Outliers are only left out of the fit; their residuals are kept. The live monitor is the
*RANSAC monitor* tab of the analysis window (section 10).

**Table** (correction G, SEM → design): Tx, Ty in nm; Mx, My in ppm (= nm per mm of distance);
rotation and orthogonality in degrees. *edge (nm)*: how far that term alone moves the furthest
contact, comparable to the threshold and the residuals.

**Linear plots in the monitor** (raw error SEM − design vs position; opposite sign to G):
dx vs x and dy vs y tilt with a **scale** error; dx vs y and dy vs x tilt in **opposite**
directions for a **rotation** and in the **same** direction for an **orthogonality** (skew)
error; a vertical offset is a **shift**. Green band around the blue model line = inliers (its
width = noise); red points far off = outliers; a curved trend = distortion an affine cannot fit.

In [ ]:
contacts = merged[RANSAC_PLACEMENT]
if len(contacts.count) < 3:
    raise RuntimeError(f"Only {len(contacts.count)} measured contact(s) in placement '{RANSAC_PLACEMENT}': "
                       "see the not-measured tiles in section 3 and the diagnostic viewer (3b)")
result = ransac_affine(contacts.sem_nm, contacts.design_nm, RANSAC_THRESHOLD_NM, seed=RANSAC_SEED)

print(f"{result.iterations} search iterations; best sample {result.best_sample.tolist()}")
print(f"Inliers: {result.inliers.sum()} / {len(result.inliers)} ({result.inliers.mean():.1%}), "
      f"reference (design centroid): x {result.reference[0] / 1000:.4f} um, y {result.reference[1] / 1000:.4f} um")
display(pd.DataFrame(report_terms(result.model, contacts.design_nm - result.reference),
                     columns=["term (correction G)", "value", "edge (nm)"]).set_index("term (correction G)"))
print("G = [[a, b, tx], [c, d, ty]] about the reference point:")
print(np.array2string(result.model[:2], precision=9, suppress_small=False))
pd.DataFrame({"inliers": error_summary(result.residuals[result.inliers]),
              "all contacts": error_summary(result.residuals)}).round(3)

## 6. Registration error after the RANSAC affine

The final registration error of every contact: its residual after the global affine G,
G(SEM) − design (`result.residuals`), outliers included. The table lists the mean dx, dy of each row of contacts
(rows grouped by design y, `ROW_GAP_NM`), after the affine and, for reference, of the stitched
contacts without any affine (SEM − design). Both are plotted in the *Registration* tabs
(section 10).

In [ ]:
registration_error = result.residuals  # (N, 2) G(SEM) - design per merged contact, nm (outliers included)
row_y, row_mean, row_count = row_means(contacts.design_nm[:, 1], registration_error, ROW_GAP_NM)
_, raw_row_mean, _ = row_means(contacts.design_nm[:, 1], contacts.error_nm, ROW_GAP_NM)  # stitched, no affine
print(f"{len(row_y)} rows, {row_count.min()}..{row_count.max()} contacts per row")
pd.DataFrame({"row_y_um": (row_y - result.reference[1]) / 1000, "contacts": row_count,
              "dx_mean_nm": row_mean[:, 0], "dy_mean_nm": row_mean[:, 1],
              "no_affine_dx_mean_nm": raw_row_mean[:, 0], "no_affine_dy_mean_nm": raw_row_mean[:, 1]}).round(3)

## 7. Moving-window affine, compared with the RANSAC affine

Instead of one affine for the whole field, `moving_window_affine` fits a plain least-squares affine
(no RANSAC, all contacts) in a window `MOVING_WINDOW_UM` tall along y, moves the window up by
`MOVING_STEP_UM`, and fits again, from the bottom of the field to the top. Each contact is
corrected by the window whose centre is nearest to its design y, so its residual
G_window(SEM) − design comes from a fit centred on it. Same sign as section 6.

The first cell compares the summaries and the per-row means with the RANSAC result and lists
the affine terms of every window (correction G, as in section 5; *edge* columns left out). Both
registration errors are tabs of the analysis window (section 10), with the same reference point.

In [ ]:
moving = moving_window_affine(contacts.sem_nm, contacts.design_nm, MOVING_WINDOW_UM * 1000, MOVING_STEP_UM * 1000)
print(f"{len(moving.centres_nm)} windows, {moving.counts.min()}..{moving.counts.max()} contacts per window")
display(pd.DataFrame({"RANSAC inliers": error_summary(result.residuals[result.inliers]),
                      "RANSAC all contacts": error_summary(result.residuals),
                      "moving window all contacts": error_summary(moving.residuals)}).round(3))

window_terms = pd.DataFrame(
    [{label: value for label, value, _ in report_terms(model, contacts.design_nm[moving.window == k] - reference)}
     for k, (model, reference) in enumerate(zip(moving.models, moving.references))])
window_terms.insert(0, "contacts", moving.counts)
window_terms.insert(0, "centre_y_um", (moving.centres_nm - result.reference[1]) / 1000)
display(window_terms.round(6))

_, moving_row_mean, _ = row_means(contacts.design_nm[:, 1], moving.residuals, ROW_GAP_NM)
pd.DataFrame({"row_y_um": (row_y - result.reference[1]) / 1000, "contacts": row_count,
              "ransac_dx_nm": row_mean[:, 0], "moving_dx_nm": moving_row_mean[:, 0],
              "ransac_dy_nm": row_mean[:, 1], "moving_dy_nm": moving_row_mean[:, 1]}).round(3)

## 8. Row-to-row pitch along y

Rows are grouped by design y (`ROW_GAP_NM`); a row's position is the mean y of its contacts and
the pitch is the spacing to the next row up, plotted at the rows' mean design y. Three sets of the
same merged contacts (`RANSAC_PLACEMENT`): the **design**, the **stitched SEM without any affine**
and the **SEM after the RANSAC affine** G. A smooth global affine hardly changes the pitch, so
jumps show local problems: a spike on a tile edge (dotted lines, from the stage centres and FOVs)
points at stitching, a pattern repeating every tile at the per-tile imaging.

The cell lists the summary and the rows where the SEM pitch differs most from the design; the
*Row pitch* tab (section 10) plots them (top: pitch, bottom: SEM − design). A row whose contacts cover a
different x range than its neighbours picks up rotation × Δx in its mean y (mostly in the set
without affine).

In [ ]:
pitch_sets = {"stitched, no affine": contacts.sem_nm, "after RANSAC affine": contacts.design_nm + result.residuals}
mid_y, design_pitch = row_pitch(contacts.design_nm[:, 1], contacts.design_nm[:, 1], ROW_GAP_NM)
pitch_table = pd.DataFrame({"row_y_um": (mid_y - result.reference[1]) / 1000, "design_nm": design_pitch})
for name, points in pitch_sets.items():
    pitch_table[f"{name} - design (nm)"] = row_pitch(contacts.design_nm[:, 1], points[:, 1], ROW_GAP_NM)[1] - design_pitch
display(pitch_table.describe().round(3))
worst = pitch_table.iloc[:, 2:].abs().max(axis=1).nlargest(10).index
display(pitch_table.loc[worst].round(3))

tile_edges_y = np.unique(np.round(np.concatenate([centers[:, 1] - fovs[:, 1] / 2, centers[:, 1] + fovs[:, 1] / 2])))

## 9. Translation vs affine stitching

RANSAC and the moving-window affine again, now on the contacts of the **affine** stitching (same τ,
seed, window and step), and the key numbers of both stitchings side by side
(`stitching_summary`): overlap tie residuals, spread of the merged overlap contacts, RANSAC inlier
share and 3σ / max of the registration error without affine, after RANSAC and after the moving
window. The second table compares the RANSAC affine G of both. The translation side is the
`RANSAC_PLACEMENT` contacts of sections 5-8; the affine side uses the mean gauge (a common
shift or affine does not change the RANSAC residuals, only the 'no affine' numbers).

In [ ]:
affine_result = ransac_affine(affine_contacts.sem_nm, affine_contacts.design_nm, RANSAC_THRESHOLD_NM, seed=RANSAC_SEED)
affine_moving = moving_window_affine(affine_contacts.sem_nm, affine_contacts.design_nm, MOVING_WINDOW_UM * 1000,
                                     MOVING_STEP_UM * 1000)
runs = {"translation": (contacts, result, moving), "affine": (affine_contacts, affine_result, affine_moving)}
summary = {name: stitching_summary(*run, ties[name]) for name, run in runs.items()}
display(pd.DataFrame(summary).round(3))
pd.DataFrame({f"G, {name} stitching": {label: value for label, value, _ in report_terms(r.model, c.design_nm - r.reference)}
              for name, (c, r, _) in runs.items()})

## 10. Analysis window (closes with the window)

All viewers in one window, one tab each. The window blocks this notebook until it is closed.

- **Stitching** (`StitchViewer`): left, SEM images, design contours/centres, Otsu and refined
  contours/centres, overlap boxes and overlap outliers; right, the error map of the selected
  placement (dot = mean design position, line = mean error × `ARROW_SCALE`, adjustable in the
  panel; magenta rings mark a spread above `SPREAD_FLAG_NM`), sharing zoom and pan. Layers and
  single tiles can be hidden on the right.
- **Comparison**: the tables of section 9 and the mean dx, dy per row of both stitchings
  (one colour each): without affine, after the RANSAC affine, after the moving-window affine.
- **Analysis, translation stitching** / **Analysis, affine stitching**: the full analysis of each
  stitching, one inner tab each: RANSAC monitor (press Run or Step), registration error after RANSAC
  and after the moving window (each with the mean error per row **without any affine** below as a
  reference), moving-window tuner, row pitch. The tuner starts from
  `MOVING_WINDOW_UM` and `MOVING_STEP_UM`; invalid settings are reported and the last result is kept.

In [ ]:
app = pg.mkQApp("AffineRANSAC analysis")
stitch_view = StitchViewer(tiles, tile_ids, design_polygons, stitch, design_stitch, refined=STITCH_REFINED,
                           use_opengl=USE_OPENGL, errors=merged, arrow_scale=ARROW_SCALE, spread_flag_nm=SPREAD_FLAG_NM)
window = StitchingComparison(runs, summary, MOVING_WINDOW_UM, MOVING_STEP_UM, RANSAC_THRESHOLD_NM, RANSAC_SEED,
                             RANSAC_DELAY_MS, ROW_GAP_NM, tile_edges_y, stitch_view=stitch_view, use_opengl=USE_OPENGL)
window.show()
app.exec()
for name, analysis in window.analyses.items():
    print(f"Tuner ({name}), last setting: window {analysis.tuner.window.value():g} um, step {analysis.tuner.step.value():g} um")